<a href="https://colab.research.google.com/github/ssykes-eth/ETH_273-0003-00L/blob/project/Project_4/deepfake_discriminator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Project 4 — Deepfake Detection 🍎 ↔️ 🍊

Welcome back! In the previous notebook you trained a latent diffusion model that turns apples into oranges (and vice versa) and exported a zip of 100 freshly-minted deepfake fruits. Now we flip roles: instead of being the forger, we become the detective. 🕵️‍♂️

### Outline

1. **Grab the data** — pull the preprocessed fruits from the repo, upload your deepfake zip, preview real vs fake side-by-side.
2. **Train / Val / Test splits** — slice the reals 80/20, split the 100 fakes into 60/15/25.
3. **Learn the real manifold** — train a convolutional autoencoder on real images only.
4. **Reconstruction-error probe** — freeze the encoder, train a small linear head on those features to separate real from fake.
5. **Evaluate the detector** — ROC / PR curves, confusion matrix, and a gallery of correct catches, misses, and false alarms.
6. **Try other pretrained encoders** — same linear-probe recipe with a frozen ImageNet ResNet50 instead of the custom AE.



### Why a two-stage recipe?

In the real world, labeled fakes are **scarce and expensive** — you'll almost always have orders of magnitude more real images than forged ones. On top of that, generative models get **noticeably better every month**, so any detector you ship today is already drifting out of distribution by next quarter. Training a heavy end-to-end classifier from scratch on a handful of fakes is a losing battle on both fronts.

The classic fix is to split the problem in two:

- **Stage 1 — Pretrain once, on real data only.** A convolutional autoencoder reconstructs real fruit images, and its encoder learns a compressed representation of the real-image manifold. No fakes are needed here, so we can use as much real data as we have.
- **Stage 2 — Retrain cheaply, as often as needed.** Freeze the encoder, discard the decoder, and train a single linear layer (a *linear probe*) on a small balanced mix of real + fake embeddings. When a new generator shows up, you don't touch the encoder — you just grab a handful of fresh fakes and retrain the tiny head in seconds.

This is the same pattern that powers MAE-style pretraining and foundation-model evaluation: expensive self-supervised features, cheap supervised heads on top.


        Input Image
             │
             ▼
     ┌────────────────┐
     │                │
     │   Encoder      │   (shrinking spatial dims,
     │  (Conv AE)     │    increasing channels / compression)
     │                │
     └──────┬─────────┘
            │
            ▼
      Embedding Vector
        (latent z)
            │
            ▼
     ┌──────────────┐
     │  Linear Head │   (frozen encoder, train only this)
     │   (Probe)    │
     └──────┬───────┘
            │
            ▼
        Prediction
     (Real vs Fake)



### Step 1 — Grab the data 🧺

In [ ]:
#@title Pull the dataset (this might take around 5-7 minutes) { display-mode: "form" }
# === Clone repo and pull LFS files ===
!git lfs install
!git clone https://github.com/eth-bmai-fs26/project.git
%cd project
!git checkout week4/deepfake
!git lfs pull
%cd /content

DATASET_DIR = 'project/week4/deepfake/dataset'

import os
for f in ['fruits_train.pt', 'fruits_test.pt']:
    size = os.path.getsize(f'{DATASET_DIR}/{f}') / (1024**2)
    print(f"{f}: {size:.0f} MB")


In [ ]:
#@title Install dependencies and imports { display-mode: "form" }
# Deepfake Detection — Frozen Encoder + Linear Probe

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision.transforms as transforms
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.metrics import (
    confusion_matrix, classification_report,
    roc_auc_score, roc_curve,
    precision_recall_curve, average_precision_score,
)
import os, shutil, random, zipfile

# Local override: the latent diffusion pipeline emits 128x128 images, so we
# run the whole detector at 128x128 (utils.py still defines IMG_SIZE=64 for
# the older pixel-space tutorials — we deliberately ignore it here).
IMG_SIZE = 128
SEED = 42

torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')


In [ ]:
# Constants
BATCH_SIZE = 256
N_FAKES = 100
N_FAKE_TRAIN = 60
N_FAKE_VAL = 15
N_FAKE_TEST = 25

# Autoencoder pretraining
AE_EPOCHS = 100
AE_LR = 1e-3

# Linear probe
PROBE_EPOCHS = 50
PROBE_LR = 1e-3
LATENT_DIM = 128


In [ ]:
#@title Load real fruit images from the LFS dataset { display-mode: "form" }
# === Load real images directly from the preprocessed .pt files ===
# These are the same tensors the diffusion notebook trained on, already at 128x128.

train_data = torch.load(f'{DATASET_DIR}/fruits_train.pt', weights_only=False)
test_data  = torch.load(f'{DATASET_DIR}/fruits_test.pt',  weights_only=False)

real_train_all = train_data['composites'].float()   # (N, 3, 128, 128) in [0, 1]
real_test_all  = test_data['composites'].float()
CLASS_NAMES = {0: 'apple', 1: 'orange'}

print(f"Real train: {tuple(real_train_all.shape)}")
print(f"Real test:  {tuple(real_test_all.shape)}")

del train_data, test_data


In [ ]:
#@title 🎯 Load deepfake images { display-mode: "form" }
#@markdown Choose whether to use the provided example deepfakes or upload your own.

load_your_own_images = True  #@param {type:"boolean"}

FAKE_DIR = 'deepfake_imgs'

if not load_your_own_images:
    import zipfile
    if not os.path.exists(FAKE_DIR):
        with zipfile.ZipFile(f'{DATASET_DIR}/deepfake_imgs.zip', 'r') as zf:
            zf.extractall('.')
        print(f"Extracted to {FAKE_DIR}/")
    else:
        print(f"{FAKE_DIR}/ already exists, skipping extraction")
else:
    print("🎯- Upload deepfake_imgs.zip:")
    if not os.path.exists(FAKE_DIR):
        try:
            from google.colab import files
            uploaded = files.upload()
            zip_name = list(uploaded.keys())[0]
        except ImportError:
            zip_name = 'deepfake_imgs.zip'
            print(f"Not in Colab — looking for {zip_name} locally")

        with zipfile.ZipFile(zip_name, 'r') as zf:
            zf.extractall('.')
        print(f"Extracted to {FAKE_DIR}/")
    else:
        print(f"{FAKE_DIR}/ already exists, skipping upload")

# Load fake images as tensors.
# IMG_SIZE is 128 to match the latent pipeline's output — this Resize is a
# safety net in case you bring in fakes at a different resolution.
fake_files = sorted([f for f in os.listdir(FAKE_DIR) if f.endswith('.png')])
print(f"Found {len(fake_files)} fake images")

to_tensor = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
])

fake_images = []
for fname in fake_files:
    img = Image.open(os.path.join(FAKE_DIR, fname)).convert('RGB')
    fake_images.append(to_tensor(img))
fake_images = torch.stack(fake_images)

print(f"Fake images tensor: {tuple(fake_images.shape)}")


In [ ]:
#@title Preview real vs fake samples { display-mode: "form" }
# === Visualize real vs fake samples ===

n_show = 8
real_sample = real_train_all[torch.randperm(len(real_train_all))[:n_show]]

fig, axes = plt.subplots(2, n_show, figsize=(2.5 * n_show, 5))
for i in range(n_show):
    axes[0, i].imshow(real_sample[i].permute(1, 2, 0).numpy())
    axes[0, i].axis('off')
    axes[1, i].imshow(fake_images[i].permute(1, 2, 0).clamp(0, 1).numpy())
    axes[1, i].axis('off')

axes[0, 0].set_ylabel('Real', fontsize=13, rotation=0, labelpad=40, va='center')
axes[1, 0].set_ylabel('Fake', fontsize=13, rotation=0, labelpad=40, va='center')
plt.suptitle('Real vs Generated (Fake) Samples', fontsize=14)
plt.tight_layout()
plt.show()


### Step 2 — Train / Val / Test splits ✂️

Standard bookkeeping: split the real images 80/20 into train/val, reuse the full real test set as-is, and divide the 100 fakes into 60 train / 15 val / 25 test. The probe training set is deliberately small and balanced — we want to see how well the frozen encoder's features generalize from a handful of examples.


In [ ]:
# === Create train / val / test splits ===
# real_train_all and real_test_all were loaded directly from the .pt files above.

# Split real training images into train and val (80/20)
n_real_total = len(real_train_all)
n_real_val = int(n_real_total * 0.2)

indices = torch.randperm(n_real_total)
real_train = real_train_all[indices[:n_real_total - n_real_val]]
real_val = real_train_all[indices[n_real_total - n_real_val:]]

# Split fake images
fake_train = fake_images[:N_FAKE_TRAIN]
fake_val = fake_images[N_FAKE_TRAIN:N_FAKE_TRAIN + N_FAKE_VAL]
fake_test = fake_images[N_FAKE_TRAIN + N_FAKE_VAL:]

print(f'Train: {len(real_train)} real + {len(fake_train)} fake')
print(f'Val:   {len(real_val)} real + {len(fake_val)} fake')
print(f'Test:  {len(real_test_all)} real + {len(fake_test)} fake')


### 🎯 Step 3 — Learn the Real Manifold 🧠

We train a convolutional autoencoder on **real images only** — no fakes during training. The encoder compresses each 128×128×3 image into a 128-dim latent, and the decoder tries to reconstruct the original from that bottleneck. Think of this as the same idea as a Masked Autoencoder: self-supervised reconstruction as a proxy task that forces the network to learn the structure of real images.

The architecture is a stack of 5 stride-2 convs (128 → 64 → 32 → 16 → 8 → 4), mirrored on the decoder side. The final 4×4×256 = 4096 feature map gets projected down to a 128-dim latent.

In [ ]:
# === Convolutional Autoencoder ===

class ConvAutoencoder(nn.Module):
    """
    Convolutional autoencoder that learns to reconstruct real fruit images.
    The bottleneck forces a compressed representation — the learned manifold.
    Images that don't lie on this manifold (fakes) will reconstruct poorly.
    """
    def __init__(self, latent_dim):
        super().__init__()

        # Encoder: 128x128x3 -> latent_dim
        self.encoder = nn.Sequential(
            nn.Conv2d(3, 32, 4, stride=2, padding=1),    # -> 64x64x32
            nn.GroupNorm(8, 32),
            nn.GELU(),

            # 🎯 TODO: Fill with the remaining Conv2d, GroupNorm and GELU


            nn.Flatten(),                                  # -> 4096
            nn.Linear(256 * 4 * 4, latent_dim),           # -> latent_dim
        )

        # Decoder: latent_dim -> 128x128x3
        self.decoder_fc = nn.Linear(latent_dim, 256 * 4 * 4)

        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(256, 256, 4, stride=2, padding=1),  # -> 8x8
            nn.GroupNorm(8, 256),
            nn.GELU(),

            # 🎯 TODO: Fill with the remaining ConvTranspose2d, GroupNorm and GELU

            nn.ConvTranspose2d(32, 3, 4, stride=2, padding=1),     # -> 128x128
            nn.Sigmoid(),  # output in [0, 1] to match image range
        )

    def encode(self, x):
        """ Encode input images to the latent space using the convolutional encoder."""
        # 🎯 TODO: use the encoder to encode the images

    def decode(self, z):
        x = self.decoder_fc(z)
        x = x.view(-1, 256, 4, 4)
        return self.decoder(x)

    def forward(self, x):
        """ Full autoencoder pass: encode then decode."""
        # 🎯 TODO: encode and decode
        z = ...
        return ...


autoencoder = ConvAutoencoder(latent_dim=LATENT_DIM).to(device)
print(f"Autoencoder parameters: {sum(p.numel() for p in autoencoder.parameters()):,}")

In [ ]:
# === Train autoencoder on REAL images only ===
# The model learns what real images look like.
# It never sees a single fake during training.

ae_optimizer = optim.Adam(autoencoder.parameters(), lr=AE_LR, weight_decay=1e-5)

ae_train_losses = []
ae_val_losses = []

# Use the real-only train/val splits from Step 2
real_train_loader = DataLoader(
    torch.utils.data.TensorDataset(real_train),
    batch_size=BATCH_SIZE, shuffle=True
)
real_val_loader = DataLoader(
    torch.utils.data.TensorDataset(real_val),
    batch_size=BATCH_SIZE, shuffle=False
)

for epoch in range(AE_EPOCHS):
    autoencoder.train()
    epoch_loss = 0.0
    n_batches = 0
    for (images,) in real_train_loader:
        # Note: we usually have (images, labels) in dataloader, but here we only have images
        # The target for our reconstruction is the input image itself!


        images = images.to(device)
        recon = ... # 🎯 TODO: use the autoencoder
        loss = F.mse_loss(..., ...) # 🎯 TODO: fill the loss

        # 🎯 TODO: fill with the usual zero_grad, backward, and step

        epoch_loss += loss.item()
        n_batches += 1

    avg_train = epoch_loss / n_batches
    ae_train_losses.append(avg_train)

    # Validation
    autoencoder.eval()
    val_loss = 0.0
    n_val = 0
    with torch.no_grad():
        for (images,) in real_val_loader:
            images = images.to(device)
            val_loss += F.mse_loss(..., ...).item() # 🎯 TODO: compute the mse_loss similarly to what you have done earlier
            n_val += 1
    avg_val = val_loss / n_val
    ae_val_losses.append(avg_val)

    if (epoch + 1) % 10 == 0 or epoch == 0:
        print(f"Epoch {epoch+1:3d}/{AE_EPOCHS}  train_mse={avg_train:.6f}  val_mse={avg_val:.6f}")

print("Autoencoder training complete ✅")

In [ ]:
#@title Autoencoder loss curves { display-mode: "form" }
# === Autoencoder loss curves ===

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(ae_train_losses, label='Train MSE', linewidth=2)
ax.plot(ae_val_losses, label='Val MSE', linewidth=2)
ax.set_xlabel('Epoch')
ax.set_ylabel('MSE (Reconstruction Error)')
ax.set_title('Approach 2: Autoencoder Training (Real Images Only)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
#@title Reconstructions: real vs fake { display-mode: "form" }
# === Visualize: how well does the AE reconstruct real vs fake? ===

autoencoder.eval()
n_show = 6

# Pick some real and fake test images
real_samples = real_test_all[:n_show]
fake_samples = fake_test[:n_show]

with torch.no_grad():
    real_recon = autoencoder(real_samples.to(device)).cpu()
    fake_recon = autoencoder(fake_samples.to(device)).cpu()

fig, axes = plt.subplots(4, n_show, figsize=(2.5 * n_show, 10))
row_labels = ['Real\nOriginal', 'Real\nReconstructed', 'Fake\nOriginal', 'Fake\nReconstructed']

for i in range(n_show):
    axes[0, i].imshow(real_samples[i].permute(1, 2, 0).clamp(0, 1).numpy())
    axes[1, i].imshow(real_recon[i].permute(1, 2, 0).clamp(0, 1).numpy())

    real_err = F.mse_loss(real_recon[i], real_samples[i]).item()
    axes[1, i].set_title(f'MSE={real_err:.4f}', fontsize=8, color='green')

    axes[2, i].imshow(fake_samples[i].permute(1, 2, 0).clamp(0, 1).numpy())
    axes[3, i].imshow(fake_recon[i].permute(1, 2, 0).clamp(0, 1).numpy())

    fake_err = F.mse_loss(fake_recon[i], fake_samples[i]).item()
    axes[3, i].set_title(f'MSE={fake_err:.4f}', fontsize=8, color='red')

for row in range(4):
    for i in range(n_show):
        axes[row, i].axis('off')
    axes[row, 0].set_ylabel(row_labels[row], fontsize=10, rotation=0,
                             labelpad=70, va='center')

plt.suptitle('Autoencoder Reconstruction: Real vs Fake', fontsize=14)
plt.tight_layout()
plt.show()

### Step 4 — Linear Probe on AE Embeddings 🔬

With a frozen autoencoder trained on reals, we encode each image into a 128-dim latent vector and train a single linear layer on top to separate real from fake.

The encoder is fully frozen — we never update its weights. We extract the 128-d bottleneck representation for every image, z-score those features using train-real statistics, and train a `Linear(128 → 1)` head on the labeled train split.

This is standard linear probing: if the AE's latent space places real and fake images in separable regions, the linear classifier will find that boundary. If it doesn't — because the AE was trained purely to reconstruct pixels, never to distinguish fruit types — the probe won't be able to compensate. Plan D picks up exactly that question.


In [ ]:
#@title Evaluation utilities (metrics, plots, gallery) { display-mode: "form" }
# === Evaluation utilities ===

@torch.no_grad()
def evaluate_binary(labels, probs, title=''):
    """Compute and plot metrics for a binary classifier."""
    preds = (probs > 0.5).astype(int)

    print("=" * 50)
    print(f"CLASSIFICATION REPORT — {title}")
    print("=" * 50)
    print(classification_report(labels, preds,
          target_names=['Real', 'Fake'], digits=3))

    auc_score = roc_auc_score(labels, probs)
    ap = average_precision_score(labels, probs)
    print(f"ROC-AUC: {auc_score:.4f}")
    print(f"Average Precision (PR-AUC): {ap:.4f}")

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # Confusion Matrix
    cm = confusion_matrix(labels, preds)
    im = axes[0].imshow(cm, interpolation='nearest', cmap='Blues')
    axes[0].set_title('Confusion Matrix', fontsize=13)
    axes[0].set_xlabel('Predicted')
    axes[0].set_ylabel('Actual')
    axes[0].set_xticks([0, 1])
    axes[0].set_yticks([0, 1])
    axes[0].set_xticklabels(['Real', 'Fake'])
    axes[0].set_yticklabels(['Real', 'Fake'])
    for i in range(2):
        for j in range(2):
            axes[0].text(j, i, str(cm[i, j]),
                        ha='center', va='center', fontsize=16,
                        color='white' if cm[i, j] > cm.max()/2 else 'black')
    fig.colorbar(im, ax=axes[0], fraction=0.046)

    # ROC Curve
    fpr, tpr, _ = roc_curve(labels, probs)
    axes[1].plot(fpr, tpr, linewidth=2, label=f'AUC = {auc_score:.3f}')
    axes[1].plot([0, 1], [0, 1], 'k--', alpha=0.3)
    axes[1].set_xlabel('False Positive Rate')
    axes[1].set_ylabel('True Positive Rate')
    axes[1].set_title('ROC Curve', fontsize=13)
    axes[1].legend(fontsize=12)
    axes[1].grid(True, alpha=0.3)

    # Precision-Recall Curve
    precision, recall, _ = precision_recall_curve(labels, probs)
    axes[2].plot(recall, precision, linewidth=2, label=f'AP = {ap:.3f}')
    axes[2].set_xlabel('Recall')
    axes[2].set_ylabel('Precision')
    axes[2].set_title('Precision-Recall Curve', fontsize=13)
    axes[2].legend(fontsize=12)
    axes[2].grid(True, alpha=0.3)

    plt.suptitle(f'{title} — Test Set Performance', fontsize=15, y=1.02)
    plt.tight_layout()
    plt.show()

    return probs, preds, auc_score, ap


def show_gallery(test_ds, probs, preds, labels, title='', n_per_row=5):
    """Prediction gallery with a proper header row per category.
    """
    fake_mask = labels == 1
    real_mask = labels == 0

    tp_idx = np.where((preds == 1) & fake_mask)[0]
    fn_idx = np.where((preds == 0) & fake_mask)[0]
    fp_idx = np.where((preds == 1) & real_mask)[0]

    categories = [
        ('Correctly detected fakes (TP)', tp_idx, 'tab:green',  'high'),
        ('Missed fakes (FN)',             fn_idx, 'tab:red',    'low'),
        ('False alarms (FP)',             fp_idx, 'tab:orange', 'high'),
    ]
    n_rows = len(categories)

    # Grid: two sub-rows per category (header strip + image strip).
    # height_ratios makes the header strip thin and the image strip tall.
    fig = plt.figure(figsize=(3 * n_per_row, 3.6 * n_rows))
    gs = fig.add_gridspec(
        nrows=n_rows * 2,
        ncols=n_per_row,
        height_ratios=[0.18, 1.0] * n_rows,
        hspace=0.35,
        wspace=0.1,
    )

    for row, (cat_title, indices, color, sort_by) in enumerate(categories):
        # ---- Header strip spanning all columns ----
        header_ax = fig.add_subplot(gs[row * 2, :])
        header_ax.text(
            0.5, 0.5,
            f'{cat_title}   —   n = {len(indices)}',
            ha='center', va='center',
            fontsize=14, fontweight='bold', color=color,
            transform=header_ax.transAxes,
        )
        header_ax.axis('off')

        # ---- Select which examples to display ----
        if len(indices) > 0:
            if sort_by == 'high':
                order = np.argsort(-probs[indices])   # most confident first
            else:
                order = np.argsort(probs[indices])    # least confident first (narrowest misses)
            sorted_idx = indices[order]
        else:
            sorted_idx = indices
        n_show = min(n_per_row, len(sorted_idx))

        # ---- Image strip ----
        for col in range(n_per_row):
            ax = fig.add_subplot(gs[row * 2 + 1, col])
            if col < n_show:
                idx = sorted_idx[col]
                img = test_ds.images[idx]
                ax.imshow(img.permute(1, 2, 0).clamp(0, 1).numpy())
                ax.set_title(f'p={probs[idx]:.2f}', fontsize=10, color=color)
            else:
                ax.text(
                    0.5, 0.5, '—',
                    ha='center', va='center',
                    transform=ax.transAxes,
                    fontsize=24, color='lightgray',
                )
            ax.set_xticks([])
            ax.set_yticks([])
            for spine in ax.spines.values():
                spine.set_visible(False)

    fig.suptitle(f'{title} — Prediction Gallery', fontsize=15, y=0.995)
    plt.show()


In [ ]:
# === Linear probe on frozen AE embeddings ===

class LinearProbe(nn.Module):
    def __init__(self, in_dim):
        super().__init__()
        self.head = ... # 🎯 TODO: fill with a linear layer which takes in in_dim and returns a single number for the classification task

    def forward(self, z):
        return self.head(z)


# Freeze the AE — we never update it again
autoencoder.eval()
for p in autoencoder.parameters():
    p.requires_grad = False

@torch.no_grad()
def extract_latents(images, batch_size=64):
    """Run images through the frozen encoder, return (N, LATENT_DIM) on CPU."""
    zs = []
    for i in range(0, len(images), batch_size):
        # Note: we are using the autoencoder's encoder to extract latent representations of the images
        zs.append(autoencoder.encode(images[i:i+batch_size].to(device)).cpu())
    return torch.cat(zs)

z_real_train = extract_latents(real_train)
z_fake_train = extract_latents(fake_train)
z_real_val   = extract_latents(real_val)
z_fake_val   = extract_latents(fake_val)
z_real_test  = extract_latents(real_test_all)
z_fake_test  = extract_latents(fake_test)

In [ ]:
#@title Normalize latents and create dataset{ display-mode: "form" }
# Z-score using train-real statistics so the probe is sensitive to
# relative deviations rather than absolute latent magnitudes.
feat_mean = z_real_train.mean(dim=0, keepdim=True)
feat_std  = z_real_train.std(dim=0, keepdim=True) + 1e-6

def _norm(z):
    return (z - feat_mean) / feat_std

z_real_train = _norm(z_real_train)
z_fake_train = _norm(z_fake_train)
z_real_val   = _norm(z_real_val)
z_fake_val   = _norm(z_fake_val)
z_real_test  = _norm(z_real_test)
z_fake_test  = _norm(z_fake_test)

In [ ]:
probe = LinearProbe(in_dim=LATENT_DIM).to(device)

print(f'Latent dim: {z_real_train.shape[1]}')
print(f'Probe parameters: {sum(p.numel() for p in probe.parameters()):,}')
print(f'AE parameters (frozen): {sum(p.numel() for p in autoencoder.parameters()):,}')

# Balance training set: downsample real to match fake count
n_probe = len(z_fake_train)
perm = torch.randperm(len(z_real_train))[:n_probe]
probe_z = torch.cat([z_real_train[perm], z_fake_train])
probe_y = torch.cat([torch.zeros(n_probe), torch.ones(n_probe)])
idx = torch.randperm(len(probe_z))
probe_z, probe_y = probe_z[idx], probe_y[idx]

probe_loader = DataLoader(
    torch.utils.data.TensorDataset(probe_z, probe_y),
    batch_size=32, shuffle=True,
)
val_z = torch.cat([z_real_val, z_fake_val])
val_y = torch.cat([torch.zeros(len(z_real_val)), torch.ones(len(z_fake_val))])

probe_opt = optim.Adam(probe.parameters(), lr=PROBE_LR)
probe_criterion = nn.BCEWithLogitsLoss()

probe_train_losses, probe_val_losses = [], []
for epoch in range(PROBE_EPOCHS):
    probe.train()
    epoch_loss = 0
    for zb, yb in probe_loader:
        zb, yb = zb.to(device), yb.to(device)
        loss = probe_criterion(probe(zb).squeeze(), yb)
        probe_opt.zero_grad()
        loss.backward()
        probe_opt.step()
        epoch_loss += loss.item()
    probe_train_losses.append(epoch_loss / len(probe_loader))

    probe.eval()
    with torch.no_grad():
        val_logits = probe(val_z.to(device)).squeeze()
        probe_val_losses.append(probe_criterion(val_logits, val_y.to(device)).item())

    if (epoch + 1) % 10 == 0 or epoch == 0:
        print(f'Probe epoch {epoch+1}/{PROBE_EPOCHS}  train={probe_train_losses[-1]:.4f}  val={probe_val_losses[-1]:.4f}')

print('Linear probe training complete ✅')


### Step 5 — Evaluate the Detector 📊

Time for the verdict. We run the frozen-encoder + linear-probe pipeline on the held-out test set (all real test images plus the 25 untouched fakes) and look at ROC-AUC, PR-AUC, the confusion matrix, and a gallery of the probe's most confident catches, its worst misses, and its false alarms.


In [ ]:
#@title Evaluate thelinear probe { display-mode: "form" }

probe.eval()
test_z = torch.cat([z_real_test, z_fake_test])
test_y = torch.cat([torch.zeros(len(z_real_test)), torch.ones(len(z_fake_test))])

with torch.no_grad():
    test_logits = probe(test_z.to(device)).squeeze().cpu()
    probe_probs_np = torch.sigmoid(test_logits).numpy()
    probe_labels_np = test_y.numpy()

probe_probs_out, probe_preds, probe_auc, probe_ap = evaluate_binary(
    probe_labels_np, probe_probs_np,
    "Approach 2: Frozen Encoder + Linear Probe"
)


In [ ]:
#@title Prediction gallery { display-mode: "form" }
# === Linear probe gallery ===

class SimpleDataset:
    def __init__(self, real_imgs, fake_imgs):
        self.images = torch.cat([real_imgs, fake_imgs], dim=0)

probe_test_ds = SimpleDataset(real_test_all, fake_test)
show_gallery(probe_test_ds, probe_probs_out, probe_preds, probe_labels_np.astype(int),
             "Frozen Encoder + Linear Probe")


## Discussion

**What does the linear probe's performance tell us about the AE's latent space?**

**If we swaps the custom AE for a different encoder and results improve substantially, what would this say about our custom AE or its training objective?**

### Same recipe, different encoder

Let's see if swapping to an encoder trained on a much larger dataset changes that. We borrow the frozen features of an ImageNet-pretrained ResNet18 and fit a single linear layer on top with the 60 labeled fakes we already have — no fine-tuning, no new data.


In [ ]:
# === Frozen ResNet18 + linear probe (logistic regression) ===
from torchvision.models import resnet18, ResNet18_Weights
from sklearn.linear_model import LogisticRegression

# ---- Frozen ImageNet ResNet18 as feature extractor ----
_resnet = resnet18(weights=ResNet18_Weights.DEFAULT).to(device).eval()
for p in _resnet.parameters():
    p.requires_grad = False
_resnet.fc = nn.Identity()  # forward(x) -> (B, 512)

_IM_MEAN = torch.tensor([0.485, 0.456, 0.406], device=device).view(1, 3, 1, 1)
_IM_STD  = torch.tensor([0.229, 0.224, 0.225], device=device).view(1, 3, 1, 1)

@torch.no_grad()
def extract_resnet_features(images, batch_size=32):
    """Resize 128->224, ImageNet-normalize, extract (N, 512) avgpool features."""
    feats = []
    for i in range(0, len(images), batch_size):
        x = images[i:i+batch_size].to(device)
        x = F.interpolate(x, size=224, mode='bilinear', align_corners=False)
        x = (x - _IM_MEAN) / _IM_STD
        feats.append(_resnet(x).cpu())
    return torch.cat(feats)

In [ ]:
#@title Extract ResNet18 features { display-mode: "form" }
print("Extracting ResNet18 features...")
f_real_train = extract_resnet_features(real_train)
f_fake_train = extract_resnet_features(fake_train)
f_real_test  = extract_resnet_features(real_test_all)
f_fake_test  = extract_resnet_features(fake_test)
print(f"  real_train: {tuple(f_real_train.shape)}")
print(f"  fake_train: {tuple(f_fake_train.shape)}")
print(f"  real_test:  {tuple(f_real_test.shape)}")
print(f"  fake_test:  {tuple(f_fake_test.shape)}")


In [ ]:
# ---- 🎯 Logistic regression linear probe ----
X_train = torch.cat([f_real_train, f_fake_train]).numpy()
y_train = np.concatenate([
    np.zeros(len(f_real_train)),
    np.ones(len(f_fake_train)),
]).astype(int)
print(f"Train: {X_train.shape}  ({(y_train==0).sum()} real + {(y_train==1).sum()} fake)")

probe = LogisticRegression(
    # 🎯 TODO: fill with appropriate parameters
)

# 🎯 TODO: fit the model

print(f"Training accuracy: {probe.score(X_train, y_train):.4f}")

In [ ]:
#@title Evaluate ResNet18 probe { display-mode: "form" }
# ---- Evaluate on test set ----
X_test = torch.cat([f_real_test, f_fake_test]).numpy()
y_test = np.concatenate([
    np.zeros(len(f_real_test)),
    np.ones(len(f_fake_test)),
]).astype(int)

plan_d_probs = probe.predict_proba(X_test)[:, 1]  # probability of "fake"

plan_d_probs_out, plan_d_preds, plan_d_auc, plan_d_ap = evaluate_binary(
    y_test, plan_d_probs, "Plan D: ResNet18 + Logistic Regression"
)

# Operating-point sweep (threshold from train-real percentiles)
train_real_probs = probe.predict_proba(f_real_train.numpy())[:, 1]
print()
print("Operating points across FPR budgets (threshold from train-real percentiles):")
print(f"  {'FPR budget':>12}  {'tau':>8}  {'test FPR':>10}  {'test recall':>12}  {'fake prec':>10}")
for budget in [0.01, 0.02, 0.05, 0.10, 0.20]:
    t = np.quantile(train_real_probs, 1.0 - budget)
    preds_b = (plan_d_probs > t).astype(int)
    tp = ((preds_b == 1) & (y_test == 1)).sum()
    fp = ((preds_b == 1) & (y_test == 0)).sum()
    rec = tp / max((y_test == 1).sum(), 1)
    fpr = fp / max((y_test == 0).sum(), 1)
    prec = tp / max(tp + fp, 1)
    print(f"  {budget:>12.2f}  {t:>8.3f}  {fpr:>10.3f}  {rec:>12.3f}  {prec:>10.3f}")

# Gallery
plan_d_test_ds = SimpleDataset(real_test_all, fake_test)
show_gallery(plan_d_test_ds, plan_d_probs_out, plan_d_preds, y_test,
             "Plan D: ResNet18 + Logistic Regression")
